lstm

In [1]:
import pandas as pd
import numpy as np

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, QED
from rdkit.Chem import rdMolDescriptors
from pathlib import Path


In [2]:
# Load generated molecules
ROOT = Path("..")
DATA = ROOT / "data"

df = pd.read_csv(DATA / "generated_molecules_lstm.csv")

print("Total molecules:", len(df))
df.head()


Total molecules: 150


,SMILES,valid,MolWt,QED,LogP,target_query,retrieved_examples
0,COC1=C(SC(=C1)C(=O)O)O,True,174.177,0.705577,1.1605,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
1,C1=CC=C(C=C1)C(=C2C(=O)C3=CC4=C3N6)CC3)OC4=CC=...,False,NaN,NaN,NaN,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
2,C1=CC(=C(C(=C1)C=O)C)C(=O)NC2=O,False,NaN,NaN,NaN,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
3,C1=CC=C(C(=C1)C2=C(C=C3C=C3)C2=C3C=CC4=CC=CC=C...,False,NaN,NaN,NaN,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
4,COC1=CC=C(C(=C1)C(=O)O)Br,True,231.045,0.847620,2.1559,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...


In [3]:
# Basic validity statistics (VERY IMPORTANT)
valid_ratio = df["valid"].mean()

print(f"Validity ratio: {valid_ratio:.2%}")
print(f"Valid molecules: {df['valid'].sum()}")
print(f"Invalid molecules: {(~df['valid']).sum()}")


Validity ratio: 71.33%
Valid molecules: 107
Invalid molecules: 43


In [4]:
# Filter valid molecules only
df_valid = df[df["valid"]].copy()
df_valid.reset_index(drop=True, inplace=True)

df_valid.head()


,SMILES,valid,MolWt,QED,LogP,target_query,retrieved_examples
0,COC1=C(SC(=C1)C(=O)O)O,True,174.177,0.705577,1.1605,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
1,COC1=CC=C(C(=C1)C(=O)O)Br,True,231.045,0.847620,2.1559,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
2,COC1=CC=CC=C1N,True,123.155,0.570691,1.2774,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
3,C1=CC2=C(C(=C1Br)Br)C=C(O2)N(C)C2=CC=CC(=C2)OC...,True,501.174,0.306069,6.8782,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
4,COC1=CC(=C(C(=C1)C(=O)O)O)O,True,184.147,0.591647,0.8046,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...


In [5]:
# Chemical property distributions
summary = df_valid[["MolWt", "LogP", "QED"]].describe()
summary


,MolWt,LogP,QED
count,107.000000,107.000000,107.000000
mean,201.338431,1.943834,0.617046
std,72.967043,1.395558,0.145199
min,88.106000,-0.965200,0.270222
25%,147.627000,1.156100,0.542802
50%,194.230000,1.663000,0.623943
75%,246.191000,2.798950,0.722233
max,501.174000,6.878200,0.877710


In [6]:
# Lipinski Rule-of-Five check
def lipinski_ok(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)

    return (
        mw <= 500 and
        logp <= 5 and
        hbd <= 5 and
        hba <= 10
    )

df_valid["Lipinski"] = df_valid["SMILES"].apply(lipinski_ok)

print("Lipinski-pass ratio:", df_valid["Lipinski"].mean())


Lipinski-pass ratio: 0.9626168224299065


In [7]:
# Uniqueness (VERY IMPORTANT)
unique_smiles = df_valid["SMILES"].nunique()
total_valid = len(df_valid)

print("Unique valid molecules:", unique_smiles)
print("Uniqueness ratio:", unique_smiles / total_valid)


Unique valid molecules: 103
Uniqueness ratio: 0.9626168224299065


In [8]:
# Novelty vs retrieved examples
retrieved_sets = set(
    "|".join(df["retrieved_examples"].dropna().tolist()).split("|")
)

df_valid["Novel"] = ~df_valid["SMILES"].isin(retrieved_sets)

print("Novelty ratio:", df_valid["Novel"].mean())


Novelty ratio: 1.0


In [9]:
# Tanimoto similarity (diversity check)
def tanimoto(sm1, sm2):
    m1, m2 = Chem.MolFromSmiles(sm1), Chem.MolFromSmiles(sm2)
    if m1 is None or m2 is None:
        return None
    fp1 = Chem.RDKFingerprint(m1)
    fp2 = Chem.RDKFingerprint(m2)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

pairs = []
smiles = df_valid["SMILES"].tolist()

for i in range(min(50, len(smiles)-1)):
    sim = tanimoto(smiles[i], smiles[i+1])
    if sim is not None:
        pairs.append(sim)

print("Average Tanimoto similarity:", np.mean(pairs))


Average Tanimoto similarity: 0.12155751759384836


lstm

In [10]:
# Final evaluation table (for report / paper)
final_metrics = {
    "Total generated": len(df),
    "Validity (%)": round(valid_ratio * 100, 2),
    "Uniqueness (%)": round((unique_smiles / total_valid) * 100, 2),
    "Lipinski pass (%)": round(df_valid["Lipinski"].mean() * 100, 2),
    "Novelty (%)": round(df_valid["Novel"].mean() * 100, 2),
    "Avg QED": round(df_valid["QED"].mean(), 3),
}

pd.DataFrame([final_metrics])


,Total generated,Validity (%),Uniqueness (%),Lipinski pass (%),Novelty (%),Avg QED
0,150,71.33,96.26,96.26,100.0,0.617


In [11]:
# Save evaluated results
OUT = DATA / "generated_molecules_evaluated_LSTM.csv"
df_valid.to_csv(OUT, index=False)
print("Saved:", OUT)


Saved: ..\data\generated_molecules_evaluated_LSTM.csv


gru

In [12]:
# Load generated molecules
ROOT = Path("..")
DATA = ROOT / "data"

df = pd.read_csv(DATA / "generated_molecules_GRU.csv")

print("Total molecules:", len(df))
df.head()


Total molecules: 15


,SMILES,valid,MolWt,QED,LogP,target_query,retrieved_examples
0,CC(C)CC1=CC=C=C(C=C1)CCCC2=C(C2)C3=CC=CC=C3,True,302.461,0.495272,6.6380,EGFR inhibitor,CCC1=C2COC(=O)C2=C(S1)C3=CC=CC=C3.CCC1=C2COC(=...
1,CC(C)CC(=O)C(=O)C1=CC=C(C=C1)CC3=CC=C(C=C2)C3=...,False,NaN,NaN,NaN,EGFR inhibitor,CCC1=C2COC(=O)C2=C(S1)C3=CC=CC=C3.CCC1=C2COC(=...
2,COC(=O)C,True,74.079,0.382967,0.1793,EGFR inhibitor,CCC1=C2COC(=O)C2=C(S1)C3=CC=CC=C3.CCC1=C2COC(=...
3,CC(CC1)N,False,NaN,NaN,NaN,EGFR inhibitor,CCC1=C2COC(=O)C2=C(S1)C3=CC=CC=C3.CCC1=C2COC(=...
4,CCC1=CC=C(C=C1OC=C(C=CC2=CC=C3)C3=CC=CC(=C3C=C...,False,NaN,NaN,NaN,EGFR inhibitor,CCC1=C2COC(=O)C2=C(S1)C3=CC=CC=C3.CCC1=C2COC(=...


In [13]:
# Basic validity statistics (VERY IMPORTANT)
valid_ratio = df["valid"].mean()

print(f"Validity ratio: {valid_ratio:.2%}")
print(f"Valid molecules: {df['valid'].sum()}")
print(f"Invalid molecules: {(~df['valid']).sum()}")


Validity ratio: 40.00%
Valid molecules: 6
Invalid molecules: 9


In [14]:
# Filter valid molecules only
df_valid = df[df["valid"]].copy()
df_valid.reset_index(drop=True, inplace=True)

df_valid.head()


,SMILES,valid,MolWt,QED,LogP,target_query,retrieved_examples
0,CC(C)CC1=CC=C=C(C=C1)CCCC2=C(C2)C3=CC=CC=C3,True,302.461,0.495272,6.6380,EGFR inhibitor,CCC1=C2COC(=O)C2=C(S1)C3=CC=CC=C3.CCC1=C2COC(=...
1,COC(=O)C,True,74.079,0.382967,0.1793,EGFR inhibitor,CCC1=C2COC(=O)C2=C(S1)C3=CC=CC=C3.CCC1=C2COC(=...
2,C1=C(C=CC=C1Cl),True,112.559,0.483383,2.3400,Serotonin receptor antagonist,B1(C2=CC=CC=C2C3=CC=CC=C31)C4=CC=CC=C4C5=CC=CC...
3,C1=C(C=CC=C1)C2=CC=C(C=C2O),True,170.211,0.696938,3.0592,Serotonin receptor antagonist,B1(C2=CC=CC=C2C3=CC=CC=C31)C4=CC=CC=C4C5=CC=CC...
4,C1=C(C(=NN=C1)C(=O)O),True,124.099,0.579387,0.1748,Serotonin receptor antagonist,B1(C2=CC=CC=C2C3=CC=CC=C31)C4=CC=CC=C4C5=CC=CC...


In [15]:
# Chemical property distributions
summary = df_valid[["MolWt", "LogP", "QED"]].describe()
summary


,MolWt,LogP,QED
count,6.000000,6.000000,6.000000
mean,151.094000,2.278117,0.534773
std,80.253352,2.427052,0.106606
min,74.079000,0.174800,0.382967
25%,115.208000,0.453825,0.486355
50%,123.627000,1.808700,0.532982
75%,158.683000,2.879400,0.577213
max,302.461000,6.638000,0.696938


In [16]:
# Lipinski Rule-of-Five check
def lipinski_ok(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)

    return (
        mw <= 500 and
        logp <= 5 and
        hbd <= 5 and
        hba <= 10
    )

df_valid["Lipinski"] = df_valid["SMILES"].apply(lipinski_ok)

print("Lipinski-pass ratio:", df_valid["Lipinski"].mean())


Lipinski-pass ratio: 0.8333333333333334


In [17]:
# Uniqueness (VERY IMPORTANT)
unique_smiles = df_valid["SMILES"].nunique()
total_valid = len(df_valid)

print("Unique valid molecules:", unique_smiles)
print("Uniqueness ratio:", unique_smiles / total_valid)


Unique valid molecules: 6
Uniqueness ratio: 1.0


In [18]:
# Novelty vs retrieved examples
retrieved_sets = set(
    "|".join(df["retrieved_examples"].dropna().tolist()).split("|")
)

df_valid["Novel"] = ~df_valid["SMILES"].isin(retrieved_sets)

print("Novelty ratio:", df_valid["Novel"].mean())


Novelty ratio: 1.0


In [19]:
# Tanimoto similarity (diversity check)
def tanimoto(sm1, sm2):
    m1, m2 = Chem.MolFromSmiles(sm1), Chem.MolFromSmiles(sm2)
    if m1 is None or m2 is None:
        return None
    fp1 = Chem.RDKFingerprint(m1)
    fp2 = Chem.RDKFingerprint(m2)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

pairs = []
smiles = df_valid["SMILES"].tolist()

for i in range(min(50, len(smiles)-1)):
    sim = tanimoto(smiles[i], smiles[i+1])
    if sim is not None:
        pairs.append(sim)

print("Average Tanimoto similarity:", np.mean(pairs))


Average Tanimoto similarity: 0.04995952982575586


In [20]:
# Final evaluation table (for report / paper)
final_metrics = {
    "Total generated": len(df),
    "Validity (%)": round(valid_ratio * 100, 2),
    "Uniqueness (%)": round((unique_smiles / total_valid) * 100, 2),
    "Lipinski pass (%)": round(df_valid["Lipinski"].mean() * 100, 2),
    "Novelty (%)": round(df_valid["Novel"].mean() * 100, 2),
    "Avg QED": round(df_valid["QED"].mean(), 3),
}

pd.DataFrame([final_metrics])


,Total generated,Validity (%),Uniqueness (%),Lipinski pass (%),Novelty (%),Avg QED
0,15,40.0,100.0,83.33,100.0,0.535


In [21]:
# Save evaluated results
OUT = DATA / "generated_molecules_evaluated_GRU.csv"
df_valid.to_csv(OUT, index=False)
print("Saved:", OUT)


Saved: ..\data\generated_molecules_evaluated_GRU.csv


transformer

In [22]:
# Load generated molecules
ROOT = Path("..")
DATA = ROOT / "data"

df = pd.read_csv(DATA / "generated_molecules_transformer.csv")

print("Total molecules:", len(df))
df.head()


Total molecules: 150


,SMILES,valid,MolWt,QED,LogP,target_query,retrieved_examples
0,C1=CCCC1CC,True,96.173,0.439471,2.3626,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
1,COC1=CCCCCC1CCCO,True,184.279,0.726988,2.4794,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
2,C1=CC=C=C=C=CC=CCC=CCC=C1,True,194.277,0.403254,4.0265,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
3,CCOCCCCCCCCCCCCCCCCCCCCCCCCCC,True,410.771,0.128786,10.4052,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
4,COC1=CC1,True,70.091,0.445451,0.9204,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...


In [23]:
# Basic validity statistics (VERY IMPORTANT)
valid_ratio = df["valid"].mean()

print(f"Validity ratio: {valid_ratio:.2%}")
print(f"Valid molecules: {df['valid'].sum()}")
print(f"Invalid molecules: {(~df['valid']).sum()}")


Validity ratio: 50.67%
Valid molecules: 76
Invalid molecules: 74


In [24]:
# Filter valid molecules only
df_valid = df[df["valid"]].copy()
df_valid.reset_index(drop=True, inplace=True)

df_valid.head()


,SMILES,valid,MolWt,QED,LogP,target_query,retrieved_examples
0,C1=CCCC1CC,True,96.173,0.439471,2.3626,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
1,COC1=CCCCCC1CCCO,True,184.279,0.726988,2.4794,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
2,C1=CC=C=C=C=CC=CCC=CCC=C1,True,194.277,0.403254,4.0265,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
3,CCOCCCCCCCCCCCCCCCCCCCCCCCCCC,True,410.771,0.128786,10.4052,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...
4,COC1=CC1,True,70.091,0.445451,0.9204,EGFR inhibitor,CCO|COC1=C(C2=C(C(=C(C=C2C=C1)Br)O)C(=O)C3=CC=...


In [25]:
# Chemical property distributions
summary = df_valid[["MolWt", "LogP", "QED"]].describe()
summary


,MolWt,LogP,QED
count,76.000000,76.000000,76.000000
mean,221.785342,3.835680,0.366521
std,185.470584,4.587338,0.146199
min,40.065000,-0.480200,0.033558
25%,77.609250,0.965875,0.294091
50%,173.276000,2.164850,0.396211
75%,304.852500,5.868525,0.465837
max,1126.148000,30.300300,0.731081


In [26]:
# Lipinski Rule-of-Five check
def lipinski_ok(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)

    return (
        mw <= 500 and
        logp <= 5 and
        hbd <= 5 and
        hba <= 10
    )

df_valid["Lipinski"] = df_valid["SMILES"].apply(lipinski_ok)

print("Lipinski-pass ratio:", df_valid["Lipinski"].mean())


Lipinski-pass ratio: 0.7105263157894737


In [27]:
# Uniqueness (VERY IMPORTANT)
unique_smiles = df_valid["SMILES"].nunique()
total_valid = len(df_valid)

print("Unique valid molecules:", unique_smiles)
print("Uniqueness ratio:", unique_smiles / total_valid)


Unique valid molecules: 64
Uniqueness ratio: 0.8421052631578947


In [28]:
# Novelty vs retrieved examples
retrieved_sets = set(
    "|".join(df["retrieved_examples"].dropna().tolist()).split("|")
)

df_valid["Novel"] = ~df_valid["SMILES"].isin(retrieved_sets)

print("Novelty ratio:", df_valid["Novel"].mean())


Novelty ratio: 0.9473684210526315


In [29]:
# Tanimoto similarity (diversity check)
def tanimoto(sm1, sm2):
    m1, m2 = Chem.MolFromSmiles(sm1), Chem.MolFromSmiles(sm2)
    if m1 is None or m2 is None:
        return None
    fp1 = Chem.RDKFingerprint(m1)
    fp2 = Chem.RDKFingerprint(m2)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

pairs = []
smiles = df_valid["SMILES"].tolist()

for i in range(min(50, len(smiles)-1)):
    sim = tanimoto(smiles[i], smiles[i+1])
    if sim is not None:
        pairs.append(sim)

print("Average Tanimoto similarity:", np.mean(pairs))


Average Tanimoto similarity: 0.19686808902227837


In [30]:
# Final evaluation table (for report / paper)
final_metrics = {
    "Total generated": len(df),
    "Validity (%)": round(valid_ratio * 100, 2),
    "Uniqueness (%)": round((unique_smiles / total_valid) * 100, 2),
    "Lipinski pass (%)": round(df_valid["Lipinski"].mean() * 100, 2),
    "Novelty (%)": round(df_valid["Novel"].mean() * 100, 2),
    "Avg QED": round(df_valid["QED"].mean(), 3),
}

pd.DataFrame([final_metrics])


,Total generated,Validity (%),Uniqueness (%),Lipinski pass (%),Novelty (%),Avg QED
0,150,50.67,84.21,71.05,94.74,0.367


In [31]:
# Save evaluated results
OUT = DATA / "generated_molecules_evaluated_transformer.csv"
df_valid.to_csv(OUT, index=False)
print("Saved:", OUT)


Saved: ..\data\generated_molecules_evaluated_transformer.csv


selfies_lstm

In [42]:
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED
from rdkit import DataStructs


In [43]:
ROOT = Path("..")
DATA = ROOT / "data"

df = pd.read_csv(DATA / "generated_molecules_selfies_lstm.csv")

print("Total molecules:", len(df))
df.head()


Total molecules: 150


,SMILES,valid,MolWt,QED,LogP,target_query
0,C/C=C/NC(=O)C1=CC=C(C=C1)C,True,175.231,0.733514,2.25842,EGFR inhibitor
1,C1=CC=C(C=C1)C[C@H1](C)[NH3+1],True,136.218,0.624252,0.85950,EGFR inhibitor
2,C(C(=O)[O-1])NC(=O)[C@@H1](C(=O)O)[NH3+1],True,176.128,0.371202,-4.45240,EGFR inhibitor
3,CCSC[NH2+1]C,True,106.214,0.392137,-0.10980,EGFR inhibitor
4,C(C[NH1+1]CCC(=O)[O-1])C(=O)[O-1],True,159.141,0.395743,-4.38440,EGFR inhibitor


In [44]:
# Basic validity statistics (VERY IMPORTANT)
valid_ratio = df["valid"].mean()

print(f"Validity ratio: {valid_ratio:.2%}")
print(f"Valid molecules: {df['valid'].sum()}")
print(f"Invalid molecules: {(~df['valid']).sum()}")


Validity ratio: 100.00%
Valid molecules: 150
Invalid molecules: 0


In [45]:
# Filter valid molecules only
df_valid = df[df["valid"]].copy()
df_valid.reset_index(drop=True, inplace=True)

df_valid.head()


,SMILES,valid,MolWt,QED,LogP,target_query
0,C/C=C/NC(=O)C1=CC=C(C=C1)C,True,175.231,0.733514,2.25842,EGFR inhibitor
1,C1=CC=C(C=C1)C[C@H1](C)[NH3+1],True,136.218,0.624252,0.85950,EGFR inhibitor
2,C(C(=O)[O-1])NC(=O)[C@@H1](C(=O)O)[NH3+1],True,176.128,0.371202,-4.45240,EGFR inhibitor
3,CCSC[NH2+1]C,True,106.214,0.392137,-0.10980,EGFR inhibitor
4,C(C[NH1+1]CCC(=O)[O-1])C(=O)[O-1],True,159.141,0.395743,-4.38440,EGFR inhibitor


In [46]:
# Chemical property distributions
summary = df_valid[["MolWt", "LogP", "QED"]].describe()
summary


,MolWt,LogP,QED
count,150.000000,150.000000,150.000000
mean,158.377553,-0.256945,0.505958
std,62.671796,1.943193,0.136118
min,46.093000,-4.504500,0.180985
25%,107.687250,-1.480375,0.407468
50%,146.726500,-0.193265,0.479542
75%,202.073500,1.033975,0.598074
max,360.272000,6.476880,0.881873


In [47]:
# Lipinski Rule-of-Five check
def lipinski_ok(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)

    return (
        mw <= 500 and
        logp <= 5 and
        hbd <= 5 and
        hba <= 10
    )

df_valid["Lipinski"] = df_valid["SMILES"].apply(lipinski_ok)

print("Lipinski-pass ratio:", df_valid["Lipinski"].mean())


Lipinski-pass ratio: 0.9866666666666667


In [48]:
# Uniqueness (VERY IMPORTANT)
unique_smiles = df_valid["SMILES"].nunique()
total_valid = len(df_valid)

print("Unique valid molecules:", unique_smiles)
print("Uniqueness ratio:", unique_smiles / total_valid)


Unique valid molecules: 148
Uniqueness ratio: 0.9866666666666667


In [49]:
pubchem_meta = pd.read_parquet(ROOT / "embeddings" / "pubchem_metadata.parquet")

pubchem_smiles = set(
    pubchem_meta["SMILES"].dropna().astype(str)
)

df_valid["Novel"] = ~df_valid["SMILES"].isin(pubchem_smiles)

print("Novelty ratio:", df_valid["Novel"].mean())


Novelty ratio: 0.9866666666666667


In [50]:
# Tanimoto similarity (diversity check)
def tanimoto(sm1, sm2):
    m1, m2 = Chem.MolFromSmiles(sm1), Chem.MolFromSmiles(sm2)
    if m1 is None or m2 is None:
        return None
    fp1 = Chem.RDKFingerprint(m1)
    fp2 = Chem.RDKFingerprint(m2)
    return DataStructs.TanimotoSimilarity(fp1, fp2)

pairs = []
smiles = df_valid["SMILES"].tolist()

for i in range(min(50, len(smiles)-1)):
    sim = tanimoto(smiles[i], smiles[i+1])
    if sim is not None:
        pairs.append(sim)

print("Average Tanimoto similarity:", np.mean(pairs))


Average Tanimoto similarity: 0.08030215388232975


In [51]:
# Final evaluation table (for report / paper)
final_metrics = {
    "Total generated": len(df),
    "Validity (%)": round(valid_ratio * 100, 2),
    "Uniqueness (%)": round((unique_smiles / total_valid) * 100, 2),
    "Lipinski pass (%)": round(df_valid["Lipinski"].mean() * 100, 2),
    "Novelty (%)": round(df_valid["Novel"].mean() * 100, 2),
    "Avg QED": round(df_valid["QED"].mean(), 3),
}

pd.DataFrame([final_metrics])


,Total generated,Validity (%),Uniqueness (%),Lipinski pass (%),Novelty (%),Avg QED
0,150,100.0,98.67,98.67,98.67,0.506


In [52]:
# Save evaluated results
OUT = DATA / "generated_molecules_evaluated_selfies_lstm.csv"
df_valid.to_csv(OUT, index=False)
print("Saved:", OUT)


Saved: ..\data\generated_molecules_evaluated_selfies_lstm.csv
